In [0]:
from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/Sao_Paulo"
)

BRONZE_SCHEMA = "workspace.bronze"
SILVER_SCHEMA = "workspace.silver"

In [0]:
def ler_bronze(nome_tabela):
    """
    Lê uma tabela da camada Bronze.
    """
    return spark.read.table(
        f"{BRONZE_SCHEMA}.{nome_tabela}"
    )


def minutos_entre(fim, inicio):
    """
    Calcula a diferença entre dois timestamps em minutos.
    """
    return (
        F.unix_timestamp(fim)
        - F.unix_timestamp(inicio)
    ) / 60.0

In [0]:
# Dimensões utilizadas para enriquecer as fatos

dim_tecnico = (
    ler_bronze("dim_tecnico")
    .dropDuplicates(["tecnico_id"])
    .select(
        "tecnico_id",
        F.col("nome_completo").alias("tecnico_nome"),
        F.col("cargo").alias("tecnico_cargo")
    )
)

dim_instituicao = (
    ler_bronze("dim_instituicao")
    .dropDuplicates(["instituicao_id"])
    .select(
        "instituicao_id",
        F.col("nome").alias("instituicao_nome")
    )
)

dim_unidade = (
    ler_bronze("dim_unidade")
    .dropDuplicates(["unidade_id"])
    .select(
        "unidade_id",
        "instituicao_id",
        F.col("nome").alias("unidade_nome"),
        "cidade",
        "estado"
    )
    .join(
        dim_instituicao,
        "instituicao_id",
        "left"
    )
)

dim_local = (
    ler_bronze("dim_local")
    .dropDuplicates(["local_id"])
    .select(
        "local_id",
        "nome_local",
        "tipo_local"
    )
)

dim_categoria_problema = (
    ler_bronze("dim_categoria_problema")
    .dropDuplicates(["categoria_problema_id"])
    .select(
        "categoria_problema_id",
        F.col("nome").alias("categoria_problema")
    )
)

dim_categoria_equipamento = (
    ler_bronze("dim_categoria_equipamento")
    .dropDuplicates(["categoria_equipamento_id"])
    .select(
        "categoria_equipamento_id",
        F.col("nome").alias("categoria_equipamento")
    )
)

dim_status_os = (
    ler_bronze("dim_status_ordem_servico")
    .dropDuplicates(["status_id"])
    .select(
        "status_id",
        F.col("nome").alias("status_os")
    )
)

dim_status_problema = (
    ler_bronze("dim_status_problema")
    .dropDuplicates(["status_problema_id"])
    .select(
        "status_problema_id",
        F.col("nome").alias("status_problema")
    )
)

In [0]:
# Agregação das tarefas para o KPI de tarefas concluídas

tarefas_por_os = (
    ler_bronze("tarefa")
    .select(
        "ordem_servico_id",
        F.col("status_ordem_servico")
        .cast("int")
        .alias("status_tarefa")
    )
    .groupBy("ordem_servico_id")
    .agg(
        F.count("*")
        .alias("qtd_tarefas_origem"),

        F.sum(
            F.when(
                F.col("status_tarefa") == 4,
                1
            ).otherwise(0)
        ).alias("qtd_tarefas_concluidas")
    )
)

In [0]:
ordem_servico = (
    ler_bronze("fato_ordem_servico")
    .dropDuplicates(["ordem_servico_id"])
)

# Tipagem das datas
for coluna in [
    "data_criacao",
    "data_prevista",
    "data_inicio",
    "data_conclusao"
]:
    ordem_servico = ordem_servico.withColumn(
        coluna,
        F.col(coluna).cast("timestamp")
    )

ordem_servico = (
    ordem_servico
    .withColumn(
        "status_id",
        F.col("status_id").cast("int")
    )
    .withColumn(
        "qtd_tarefas",
        F.col("qtd_tarefas").cast("long")
    )
    .withColumn(
        "tempo_pausado_minutos",
        F.col("tempo_pausado_minutos").cast("double")
    )
    .withColumn(
        "tempo_resolucao_minutos",
        F.col("tempo_resolucao_minutos").cast("double")
    )
)

In [0]:
# Tempos negativos são considerados inválidos

ordem_servico = (
    ordem_servico
    .withColumn(
        "tempo_pausado_minutos",
        F.when(
            F.col("tempo_pausado_minutos") >= 0,
            F.col("tempo_pausado_minutos")
        )
    )
    .withColumn(
        "tempo_resolucao_minutos",
        F.when(
            F.col("tempo_resolucao_minutos") >= 0,
            F.col("tempo_resolucao_minutos")
        )
    )
)

In [0]:
# Regras de status utilizadas nos KPIs

ordem_servico = (
    ordem_servico
    .withColumn(
        "esta_aberta",
        F.col("status_id").isin(1, 2, 3)
    )
    .withColumn(
        "finalizada",
        F.col("status_id") == 4
    )
)

In [0]:
# Tempo de espera válido: criação até primeiro início

espera_valida = (
    F.col("data_criacao").isNotNull()
    & F.col("data_inicio").isNotNull()
    & (
        F.col("data_inicio")
        >= F.col("data_criacao")
    )
)

ordem_servico = (
    ordem_servico
    .withColumn(
        "espera_valida",
        espera_valida
    )
    .withColumn(
        "tempo_espera_minutos",
        F.when(
            espera_valida,
            minutos_entre(
                F.col("data_inicio"),
                F.col("data_criacao")
            )
        )
    )
)

In [0]:
# Datas válidas para avaliação de conclusão no prazo

datas_prazo_validas = (
    F.col("data_criacao").isNotNull()
    & F.col("data_prevista").isNotNull()
    & F.col("data_conclusao").isNotNull()
    & (
        F.col("data_conclusao")
        >= F.col("data_criacao")
    )
)

ordem_servico = (
    ordem_servico
    .withColumn(
        "datas_prazo_validas",
        datas_prazo_validas
    )
    .withColumn(
        "concluida_no_prazo_calc",
        F.when(
            datas_prazo_validas,
            F.col("data_conclusao")
            <= F.col("data_prevista")
        )
    )
)

In [0]:
# Enriquecimento da fato de OS

ordem_servico = (
    ordem_servico

    .join(
        tarefas_por_os,
        "ordem_servico_id",
        "left"
    )

    .join(
        dim_tecnico,
        "tecnico_id",
        "left"
    )

    .join(
        dim_status_os,
        "status_id",
        "left"
    )

    .join(
        dim_categoria_problema,
        "categoria_problema_id",
        "left"
    )

    .join(
        dim_unidade,
        "unidade_id",
        "left"
    )

    .join(
        dim_local,
        "local_id",
        "left"
    )

    .withColumn(
        "data_criacao_dia",
        F.to_date("data_criacao")
    )

    .withColumn(
        "data_conclusao_dia",
        F.to_date("data_conclusao")
    )
)

In [0]:
problema = (
    ler_bronze("fato_problema")
    .dropDuplicates(["problema_id"])
    .withColumn(
        "data_criacao",
        F.col("data_criacao").cast("timestamp")
    )
    .withColumn(
        "status_problema_id",
        F.col("status_problema_id").cast("int")
    )
)

In [0]:
problema = (
    problema

    .withColumn(
        "esta_pendente",
        F.col("status_problema_id") == 0
    )

    .withColumn(
        "esta_reprovado",
        F.col("status_problema_id") == 1
    )

    .withColumn(
        "esta_aprovado",
        F.col("status_problema_id") == 2
    )

    .withColumn(
        "gerou_ordem_servico",
        F.col("gerou_ordem_servico").cast("boolean")
    )

    .withColumn(
        "data_criacao_dia",
        F.to_date("data_criacao")
    )
)

In [0]:
problema = (
    problema

    .join(
        dim_status_problema,
        "status_problema_id",
        "left"
    )

    .join(
        dim_categoria_equipamento,
        "categoria_equipamento_id",
        "left"
    )

    .join(
        dim_unidade,
        "unidade_id",
        "left"
    )

    .join(
        dim_local,
        "local_id",
        "left"
    )
)

In [0]:
ocorrencia = (
    ler_bronze("fato_ocorrencia")
    .dropDuplicates(["ocorrencia_id"])

    .withColumn(
        "data_criacao",
        F.col("data_criacao").cast("timestamp")
    )

    .withColumn(
        "prioridade",
        F.col("prioridade").cast("int")
    )

    .withColumn(
        "esta_ativo",
        F.col("esta_ativo").cast("boolean")
    )

    .withColumn(
        "data_criacao_dia",
        F.to_date("data_criacao")
    )
)

In [0]:
# Padronização textual da prioridade

ocorrencia = (
    ocorrencia
    .withColumn(
        "prioridade_nome",
        F.when(
            F.col("prioridade") == 0,
            "Alta"
        )
        .when(
            F.col("prioridade") == 1,
            "Média"
        )
        .when(
            F.col("prioridade") == 2,
            "Baixa"
        )
        .otherwise("Não informada")
    )
)

In [0]:
ocorrencia = (
    ocorrencia

    .join(
        dim_tecnico,
        "tecnico_id",
        "left"
    )

    .join(
        dim_categoria_problema,
        "categoria_problema_id",
        "left"
    )

    .join(
        dim_categoria_equipamento,
        "categoria_equipamento_id",
        "left"
    )

    .join(
        dim_unidade,
        "unidade_id",
        "left"
    )

    .join(
        dim_local,
        "local_id",
        "left"
    )
)

In [0]:
print(
    "OS:",
    ordem_servico.count()
)

print(
    "Solicitações:",
    problema.count()
)

print(
    "Ocorrências:",
    ocorrencia.count()
)

OS: 160
Solicitações: 240
Ocorrências: 100


In [0]:
display(
    ordem_servico.limit(10)
)

local_id,unidade_id,categoria_problema_id,status_id,tecnico_id,ordem_servico_id,problema_id,data_criacao_id,data_prevista_id,data_inicio_id,data_conclusao_id,data_criacao,data_prevista,data_inicio,data_conclusao,prioridade,qtd_tarefas,qtd_reaberturas,qtd_reatribuicoes,tempo_pausado_minutos,tempo_espera_minutos,tempo_resolucao_minutos,esta_aberta,esta_atrasada,finalizada,foi_finalizada,concluida_no_prazo,foi_reaberta,foi_reatribuida,prioridade_nome,_ingestion_timestamp,_source_system,_source_object,espera_valida,datas_prazo_validas,concluida_no_prazo_calc,qtd_tarefas_origem,qtd_tarefas_concluidas,tecnico_nome,tecnico_cargo,status_os,categoria_problema,instituicao_id,unidade_nome,cidade,estado,instituicao_nome,nome_local,tipo_local,data_criacao_dia,data_conclusao_dia
28,5,8,0,27,28,28,20261008,20261015,null,null,2026-10-07T23:59:17.482-03:00,2026-10-14T23:59:17.482-03:00,null,null,0,3,0,0,0.0,null,null,false,false,false,false,false,false,false,Alta,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,false,false,null,3,0,Carla Cardoso,Técnico de Manutenção,Não iniciadas,Segurança,3,Unidade 5,Santo André,SP,Indústrias Aurora,Auditório 5,Auditório,2026-10-07,null
29,5,9,0,28,29,29,20261021,20261028,null,null,2026-10-20T23:59:17.482-03:00,2026-10-27T23:59:17.482-03:00,null,null,1,3,0,0,0.0,null,null,false,false,false,false,false,false,false,Média,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,false,false,null,3,0,Diego Henriques,Técnico de Manutenção,Não iniciadas,Iluminação,3,Unidade 5,Santo André,SP,Indústrias Aurora,Almoxarifado 5,Almoxarifado,2026-10-20,null
30,5,10,0,27,30,30,20261020,20261027,null,null,2026-10-19T23:59:17.482-03:00,2026-10-26T23:59:17.482-03:00,null,null,2,3,0,0,0.0,null,null,false,false,false,false,false,false,false,Baixa,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,false,false,null,3,0,Carla Cardoso,Técnico de Manutenção,Não iniciadas,Limpeza e Conservação,3,Unidade 5,Santo André,SP,Indústrias Aurora,Área Comum 5,Área Comum,2026-10-19,null
33,6,3,0,34,33,33,20261017,20261024,null,null,2026-10-16T23:59:17.482-03:00,2026-10-23T23:59:17.482-03:00,null,null,2,3,0,0,0.0,null,null,false,false,false,false,false,false,false,Baixa,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,false,false,null,3,0,João Barbosa,Técnico de Manutenção,Não iniciadas,Mecânica,3,Unidade 6,Guarulhos,SP,Indústrias Aurora,Administrativo 6,Depto. Administrativo,2026-10-16,null
93,16,3,2,94,93,93,20261010,20261025,20261010,null,2026-10-09T23:59:17.482-03:00,2026-10-24T23:59:17.482-03:00,2026-10-10T08:59:17.482-03:00,null,2,3,0,0,16365.279081766666,540.0,null,true,false,false,false,false,false,false,Baixa,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,true,false,null,3,1,William Barbosa,Técnico de Manutenção,Pausada,Mecânica,8,Unidade 16,Barueri,SP,Prefeitura de Vila Serena,Administrativo 16,Depto. Administrativo,2026-10-09,null
7,2,1,4,10,151,151,20260930,20261008,20260930,20261011,2026-09-30T00:59:17.482-03:00,2026-10-08T00:59:17.482-03:00,2026-09-30T11:59:17.482-03:00,2026-10-11T00:59:17.482-03:00,0,3,0,0,0.0,660.0,15180.0,false,false,true,true,false,false,false,Alta,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,true,true,false,3,3,João Barbosa,Técnico de Manutenção,Finalizadas,Elétrica,1,Unidade 2,Campinas,SP,Colégio Horizonte,Sala de Aula 2,Sala de Aula,2026-09-30,2026-10-11
9,2,9,0,10,9,9,20261013,20261020,null,null,2026-10-12T23:59:17.482-03:00,2026-10-19T23:59:17.482-03:00,null,null,2,3,0,0,0.0,null,null,false,false,false,false,false,false,false,Baixa,2026-10-23T08:44:32.938-03:00,postgresql,fato_ordem_servico,false,false,null,3,0,João Barbosa,Técnico de Manutenção,Não iniciadas,Iluminação,1,Unidade 2,Campinas,SP,Colégio Horizonte,Administrativo 2,Depto. Administrativo,2026-10-12,null
11,2,1,0,10,11,11,20261011,20261018,null,null,2026-10-10T23:59:17.482-03:00,2026-10-17T23:59:17.482-03:00,null,null,1,3,0,0,0.0,null,null,false,false,false,false,false,false,false,M

In [0]:
display(
    problema.limit(10)
)

local_id,unidade_id,categoria_equipamento_id,status_problema_id,problema_id,solicitante_id,data_criacao_id,data_criacao,quantidade_problema,esta_pendente,esta_aprovado,esta_reprovado,gerou_ordem_servico,_ingestion_timestamp,_source_system,_source_object,data_criacao_dia,status_problema,categoria_equipamento,instituicao_id,unidade_nome,cidade,estado,instituicao_nome,nome_local,tipo_local
28,5,8,2,28,29,20260822,2026-08-21T23:59:17.482-03:00,1,false,true,false,true,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-08-21,Aprovado,Quadros Elétricos,3,Unidade 5,Santo André,SP,Indústrias Aurora,Auditório 5,Auditório
29,5,9,2,29,30,20260815,2026-08-14T23:59:17.482-03:00,1,false,true,false,true,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-08-14,Aprovado,Bebedouros,3,Unidade 5,Santo André,SP,Indústrias Aurora,Almoxarifado 5,Almoxarifado
30,5,10,2,30,29,20260808,2026-08-07T23:59:17.482-03:00,1,false,true,false,true,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-08-07,Aprovado,Equipamentos Mecânicos,3,Unidade 5,Santo André,SP,Indústrias Aurora,Área Comum 5,Área Comum
33,6,3,2,33,36,20260718,2026-07-17T23:59:17.482-03:00,1,false,true,false,true,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-07-17,Aprovado,Iluminação,3,Unidade 6,Guarulhos,SP,Indústrias Aurora,Administrativo 6,Depto. Administrativo
93,16,3,2,93,96,20260817,2026-08-16T23:59:17.482-03:00,1,false,true,false,true,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-08-16,Aprovado,Iluminação,8,Unidade 16,Barueri,SP,Prefeitura de Vila Serena,Administrativo 16,Depto. Administrativo
7,2,1,2,151,12,20260930,2026-09-29T23:59:17.482-03:00,1,false,true,false,true,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-09-29,Aprovado,Computadores,1,Unidade 2,Campinas,SP,Colégio Horizonte,Sala de Aula 2,Sala de Aula
27,5,1,0,171,30,20260602,2026-06-01T23:59:17.482-03:00,1,true,false,false,false,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-06-01,Pendente,Computadores,3,Unidade 5,Santo André,SP,Indústrias Aurora,Administrativo 5,Depto. Administrativo
56,10,10,0,200,59,20260818,2026-08-17T23:59:17.482-03:00,1,true,false,false,false,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-08-17,Pendente,Equipamentos Mecânicos,5,Unidade 10,Barueri,SP,Escola Nova Geração,Laboratório 10,Laboratório
64,11,8,0,208,65,20260709,2026-07-08T23:59:17.482-03:00,1,true,false,false,false,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-07-08,Pendente,Quadros Elétricos,6,Unidade 11,Santo André,SP,Universidade Integra,Auditório 11,Auditório
77,13,1,1,221,78,20260821,2026-08-20T23:59:17.482-03:00,1,false,false,true,false,2026-10-23T08:44:37.238-03:00,postgresql,fato_problema,2026-08-20,Reprovado,Computadores,7,Unidade 13,São Paulo,SP,Grupo Logística Sul,Almoxarifado 13,Almoxarifado


In [0]:
display(
    ocorrencia.limit(10)
)

local_id,unidade_id,categoria_equipamento_id,categoria_problema_id,tecnico_id,ocorrencia_id,data_criacao_id,data_criacao,prioridade,esta_ativo,quantidade_ocorrencia,prioridade_nome,_ingestion_timestamp,_source_system,_source_object,data_criacao_dia,tecnico_nome,tecnico_cargo,categoria_problema,categoria_equipamento,instituicao_id,unidade_nome,cidade,estado,instituicao_nome,nome_local,tipo_local
28,5,8,8,27,28,20260725,2026-07-24T23:59:17.482-03:00,0,true,1,Alta,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-07-24,Carla Cardoso,Técnico de Manutenção,Segurança,Quadros Elétricos,3,Unidade 5,Santo André,SP,Indústrias Aurora,Auditório 5,Auditório
29,5,9,9,28,29,20260722,2026-07-21T23:59:17.482-03:00,1,true,1,Média,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-07-21,Diego Henriques,Técnico de Manutenção,Iluminação,Bebedouros,3,Unidade 5,Santo André,SP,Indústrias Aurora,Almoxarifado 5,Almoxarifado
30,5,10,10,27,30,20260719,2026-07-18T23:59:17.482-03:00,2,true,1,Baixa,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-07-18,Carla Cardoso,Técnico de Manutenção,Limpeza e Conservação,Equipamentos Mecânicos,3,Unidade 5,Santo André,SP,Indústrias Aurora,Área Comum 5,Área Comum
33,6,3,3,34,33,20260710,2026-07-09T23:59:17.482-03:00,2,true,1,Baixa,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-07-09,João Barbosa,Técnico de Manutenção,Mecânica,Iluminação,3,Unidade 6,Guarulhos,SP,Indústrias Aurora,Administrativo 6,Depto. Administrativo
93,16,3,3,94,93,20260819,2026-08-18T23:59:17.482-03:00,2,true,1,Baixa,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-08-18,William Barbosa,Técnico de Manutenção,Mecânica,Iluminação,8,Unidade 16,Barueri,SP,Prefeitura de Vila Serena,Administrativo 16,Depto. Administrativo
9,2,9,9,10,9,20260920,2026-09-19T23:59:17.482-03:00,2,true,1,Baixa,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-09-19,João Barbosa,Técnico de Manutenção,Iluminação,Bebedouros,1,Unidade 2,Campinas,SP,Colégio Horizonte,Administrativo 2,Depto. Administrativo
11,2,1,1,10,11,20260914,2026-09-13T23:59:17.482-03:00,1,true,1,Média,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-09-13,João Barbosa,Técnico de Manutenção,Elétrica,Computadores,1,Unidade 2,Campinas,SP,Colégio Horizonte,Almoxarifado 2,Almoxarifado
32,6,2,2,33,32,20260713,2026-07-12T23:59:17.482-03:00,1,true,1,Média,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-07-12,Iris Lopes,Técnico de Manutenção,Hidráulica,Ar-condicionado,3,Unidade 6,Guarulhos,SP,Indústrias Aurora,Laboratório 6,Laboratório
49,9,9,9,52,49,20260910,2026-09-09T23:59:17.482-03:00,0,true,1,Alta,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-09-09,Diego Henriques,Técnico de Manutenção,Iluminação,Bebedouros,5,Unidade 9,Osasco,SP,Escola Nova Geração,Sala de Aula 9,Sala de Aula
59,10,9,9,58,59,20260811,2026-08-10T23:59:17.482-03:00,1,true,1,Média,2026-10-23T08:44:28.851-03:00,postgresql,fato_ocorrencia,2026-08-10,João Barbosa,Técnico de Manutenção,Iluminação,Bebedouros,5,Unidade 10,Barueri,SP,Escola Nova Geração,Almoxarifado 10,Almoxarifado


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

DataFrame[]

In [0]:
(
    ordem_servico
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{SILVER_SCHEMA}.ordem_servico"
    )
)

(
    problema
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{SILVER_SCHEMA}.problema"
    )
)

(
    ocorrencia
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{SILVER_SCHEMA}.ocorrencia"
    )
)